# പാഠം 01 - എഐ ഏജന്റുകളെക്കുറിച്ചുള്ള പരിചയം

**ആരംഭക്കാർക്കുള്ള AI ഏജന്റുകൾ** കോഴ്സിലെ ആദ്യ പാഠത്തിലേക്ക് സ്വാഗതം!

ഒരു **AI ഏജന്റ്** എന്നത് ഒരു വലിയ ഭാഷാ മോഡൽ (LLM) തന്റെ വാദാസക്തി എഞ്ചിനായി ഉപയോഗിച്ച് പ്രവർത്തിക്കുന്ന ഒരു പ്രോഗ്രാമാണ്, ഇത് യാഥാർത്ഥ്യത്തിൽ *പ്രവർത്തനങ്ങൾ* നടത്താൻ കഴിയും — APIകൾ വിളിക്കുക, ഡേറ്റബേസുകൾ ചോദിക്കുക, അല്ലെങ്കിൽ കോഡ് പ്രവർത്തിപ്പിക്കുക — ഒരു ഉപയോക്താവിനായി ഒരു ലക്ഷ്യം പൂർത്തീകരിക്കാൻ.

ഈ നോട്ട്‌ബുക്കിൽ നിങ്ങൾ നിങ്ങളുടെ ആദ്യ ഏജന്റ് നിർമ്മിക്കും: അവധിക്ക് അനുയോജ്യമായ സ്ഥലങ്ങൾ ശുപാർശ ചെയ്യുന്ന ഒരു **ട്രാവൽ ഏജന്റ്**. ഇതിലൂടെ നിങ്ങൾ പഠിക്കും:

1. **Microsoft Agent Framework** ഉപയോഗിച്ച് Microsoft Foundry Agent Service കേണക്ട് ചെയ്യുന്നത്.
2. ഏജന്റ് ഒരു **ടൂൾ** നൽകുക — ഇത് കോളു ചെയ്യാൻ കഴിയുന്ന സാധാരണ പൈതൺ ഫംഗ്ഷൻ.
3. ഏജന്റ് പ്രവർത്തിപ്പിച്ച് അതിന്റെ പ്രതികരണം പരിശോധിക്കുക.
4. ഏജന്റിന്റെ പ്രതികരണം ടോക്കൺ-ബൈ-ടോക്കണായി സ്ട്രീം ചെയ്യുക.


## ക്രമീകരണം

ഈ നോട്ട്‌ബുക്ക് പ്രവർത്തിപ്പിക്കാൻ മുമ്പ്, നിങ്ങൾക്കുണ്ടെന്ന് ഉറപ്പാക്കുക:

1. **വിതരണ ചെയ്‌ത ചാറ്റ് മോഡലുള്ള ഒരു Microsoft Foundry പ്രോജക്റ്റ്** (ഉദാഹരണത്തിന് `gpt-4.1-mini`).
2. **Azure CLI ഉപയോഗിച്ച് ലോഗിൻ ചെയ്തിട്ടുള്ളതു** — നിങ്ങളുടെ ടെർമിനലിൽ `az login` പ്രവർത്തിപ്പിക്കുക.
3. **ആവശ്യമായ പരിസ്ഥിതി വ്യത്യാസങ്ങൾ സജ്ജമാക്കിയിട്ടുള്ളതു:**
   - `AZURE_AI_PROJECT_ENDPOINT` — നിങ്ങളുടെ Microsoft Foundry പ്രോജക്റ്റ് എന്റ്പോയിൻറ്.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — നിങ്ങൾ വിതരണ ചെയ്‌ത മോഡലിന്റെ പേര്.

താഴെയുള്ള സെൽ നിങ്ങൾക്ക് വേണ്ട പൈത്തൺ പാക്കേജുകൾ ഇൻസ്റ്റാൾ ചെയ്യും.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## നിങ്ങളുടെ ആദ്യ ഏജന്റ് സൃഷ്ടിക്കുന്നത്

ഒരു ഏജന്റിന് രണ്ട് കാര്യങ്ങൾ ആവശ്യമുണ്ട്:

- **നിർദ്ദേശങ്ങൾ** ഇത് *ആരാണ്* എന്നും *എങ്ങനെ പെരുമാറണം* എന്നും പറയുന്നത് (ഒരു സിസ്റ്റം പ്രോംപ്റ്റ്).
- **ഉപകരണങ്ങൾ** — ഏജന്റ് വിവരങ്ങൾ വീണ്ടെടുക്കുന്നതിനായി അല്ലെങ്കിൽ പ്രവർത്തനങ്ങൾ ചെയ്യുന്നതിനുവേണ്ടി വിളിക്കാവുന്ന `@tool` ആണ് അലങ്കരിച്ചിരിക്കുന്ന Python ഫങ്ഷനുകള്‍.

താഴെ നാം സാധാരണ വിനോദസഞ്ചാര ഗമ്യസ്ഥലങ്ങളുടെ പട്ടിക നൽകുന്ന ഒരു ലളിത ഉപകരണം നിർവചിക്കുന്നു. ഉപയോക്താവ് യാത്രാ നിർദ്ദേശങ്ങൾ ചോദിക്കുമ്പോൾ ഏജന്റ് ഈ ഉപകരണം ഉപയോഗിക്കും.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## സ്റ്റ്രീമിംഗ് പ്രതികരണങ്ങൾ

കൂടുതൽ ഇന്ററാക്ടീവ് അനുഭവത്തിനായി, നിങ്ങൾ ഏജന്റിന്റെ പ്രതികരണം **സ്റ്റ്രീം** ചെയ്യാൻ കഴിയും. മുഴുവൻ മറുപടി വരെയാണ് കാത്തിരിക്കുന്നത് പകരം, ഏജന്റ് സൃഷ്ടിക്കുന്നതിനു സമാനമായി ടെക്സ്റ്റ് ഭാഗങ്ങൾ നൽകുന്നു. ഇത് പ്രത്യേകിച്ച് ചാറ്റ് ഇന്ററ്ഫേസുകളിൽ പ്രാരംഭമായുള്ള ഔട്ട്‌പുട്ട് പ്രദർശിപ്പിക്കേണ്ട സ്ഥലങ്ങളിലാണ് ഏറെ സഹായകമാകുന്നത്.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:

- **Microsoft Foundry Agent Service-ലേക്കായി കണക്ഷൻ ഉണ്ടാക്കുന്ന ഒരു പ്രൊവൈഡർ സൃഷ്ടിക്കുക** `FoundryChatClient` ഉപയോഗിച്ച്.
- **എജന്റ് നിങ്ങളുടെ Python ഫംഗ്ഷനുകളെ കാണിക്കാനായി `@tool` ഡെക്കറേറ്റർ ഉപയോഗിച്ച് ഒരു ടൂൾ നിർവചിക്കുക**.
- **ഉപയോക്തൃ സന്ദേശം ഉപയോഗിച്ച് എജന്റ് പ്രവർത്തിപ്പിക്കുക** മറുപടിയൊന്നും പ്രിന്റ് ചെയ്യുക.
- **റിയൽ-ടൈം ഔട്ട്‌പുട്ടിനായി പ്രതികരണങ്ങൾ സ്റ്റ്രീം ചെയ്യുക**.

അടുത്ത പാഠത്തിൽ ഞങ്ങൾ ഏജന്റിക് ഫ്രെയിംവർക്കുകളെ കൂടുതൽ ആഴത്തിൽ പഠിച്ച് ഏജന്റുകൾക്ക് കൂടുതൽ ശക്തമായ ടൂളുകളും ബഹു-പടി നിര്‍ണ്ണയ ശേഷിയും നൽകേണ്ടതിനെക്കുറിച്ച് അന്വേഷിക്കും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
